In [1]:
import os
import glob
import gc
import numpy as np
from astropy.io import fits
from skimage.util import view_as_windows
from astropy.convolution import convolve, Moffat2DKernel
from astropy.visualization import AsinhStretch

# ==========================================
# 核心模組 1：正向物理退化模型 (嚴謹的線性物理域)
# ==========================================
class AstroDegrader:
    def __init__(self, psf_fwhm=3.5, exptime=1000.0, read_noise_std=0.01):
        self.psf_fwhm = psf_fwhm
        self.exptime = exptime
        self.read_noise_std = read_noise_std

        gamma = psf_fwhm / (2 * np.sqrt(2**(1/4.765) - 1))
        self.kernel = Moffat2DKernel(gamma=gamma, alpha=4.765)

    def apply_degradation(self, clean_linear_patch):
        # 1. 光學模糊
        blurred = convolve(clean_linear_patch, self.kernel, boundary='extend')
        # 2. 泊松散粒雜訊 (轉換為絕對光子數)
        expected_photons = np.clip(blurred * self.exptime, 0, None)
        noisy_photons = np.random.poisson(expected_photons)
        noisy_flux = noisy_photons / self.exptime
        # 3. 高斯讀取雜訊 (保留負值背景)
        read_noise = np.random.normal(0, self.read_noise_std, noisy_flux.shape)
        return noisy_flux + read_noise

# ==========================================
# 核心模組 2：單一檔案處理管線 (讀取 -> 切片 -> 退化 -> 存檔)
# ==========================================
def process_single_fits(fits_path, output_dir, degrader, patch_size=256, stride=128):
    base_name = os.path.splitext(os.path.basename(fits_path))[0]
    save_path = os.path.join(output_dir, f"dataset_{base_name}.npz")

    if os.path.exists(save_path):
        print(f"⏩ 已跳過: {base_name} (檔案已存在)")
        return

    print(f"\n🚀 開始處理: {base_name}")

    try:
        # --- Phase A: 記憶體友善的線性讀取與切片 ---
        with fits.open(fits_path) as hdul:
            data_ext = next((i for i, hdu in enumerate(hdul) if hdu.header.get('EXTNAME') == 'SCI'), 0)
            image = hdul[data_ext].data

        image_clean = np.ascontiguousarray(np.nan_to_num(image, nan=0.0))
        mask_float = np.ascontiguousarray(((image != 0) & np.isfinite(image)).astype(float))

        img_patches = view_as_windows(image_clean, (patch_size, patch_size), step=stride).reshape(-1, patch_size, patch_size)
        mask_patches = view_as_windows(mask_float, (patch_size, patch_size), step=stride).reshape(-1, patch_size, patch_size)

        target_linear_patches = [p_img for p_img, p_mask in zip(img_patches, mask_patches) if np.min(p_mask) != 0 and np.std(p_img) > 1e-5]

        # 釋放記憶體
        del image, image_clean, mask_float, img_patches, mask_patches
        gc.collect()

        if not target_linear_patches:
            print(f"⚠️ 警告: {base_name} 沒有產生任何有效切片。")
            return

        # --- Phase B: 物理退化與非線性同步拉伸 ---
        asinh_transform = AsinhStretch(a=0.1)
        input_norm_patches = []
        target_norm_patches = []
        
        # [新增]：用於儲存絕對物理極值的陣列
        v_min_list = []
        v_max_list = []

        for clean_linear in target_linear_patches:
            degraded_linear = degrader.apply_degradation(clean_linear)

            v_min = np.percentile(clean_linear, 0.1)
            v_max = np.percentile(clean_linear, 99.9)
            if v_max == v_min: v_max = v_min + 1e-8

            clean_norm = np.clip(asinh_transform((clean_linear - v_min) / (v_max - v_min)), -0.05, 1.05)
            degraded_norm = np.clip(asinh_transform((degraded_linear - v_min) / (v_max - v_min)), -0.05, 1.05)

            input_norm_patches.append(degraded_norm)
            target_norm_patches.append(clean_norm)
            
            # [新增]：紀錄該切片的絕對物理極值
            v_min_list.append(v_min)
            v_max_list.append(v_max)

        inputs_f32 = np.array(input_norm_patches, dtype=np.float32)
        targets_f32 = np.array(target_norm_patches, dtype=np.float32)
        
        # [新增]：將極值轉換為 float32 陣列
        v_mins_f32 = np.array(v_min_list, dtype=np.float32)
        v_maxs_f32 = np.array(v_max_list, dtype=np.float32)

        # --- Phase C: 存檔與 Metadata 封裝 ---
        metadata = {'psf_fwhm': degrader.psf_fwhm, 'exptime': degrader.exptime, 'read_noise': degrader.read_noise_std, 'asinh_a': 0.1}
        
        # [修改]：將 v_mins 和 v_maxs 寫入 npz 檔
        np.savez_compressed(
            save_path, 
            inputs=inputs_f32, 
            targets=targets_f32, 
            v_mins=v_mins_f32,   # 寫入 v_min
            v_maxs=v_maxs_f32,   # 寫入 v_max
            metadata=metadata
        )

        print(f"✅ 成功儲存 {inputs_f32.shape[0]} 張成對切片與極值至 -> {save_path}")

    except Exception as e:
        print(f"❌ 處理 {base_name} 時發生錯誤: {e}")

# ==========================================
# 執行區：啟動自動化資料夾掃描
# ==========================================
if __name__ == "__main__":
    INPUT_DIR = "raw_fits"
    OUTPUT_DIR = "processed_dataset"

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    my_degrader = AstroDegrader(psf_fwhm=3.5, exptime=1000.0, read_noise_std=0.01)

    fits_files = glob.glob(os.path.join(INPUT_DIR, "*.fits"))
    print(f"🔍 總共找到 {len(fits_files)} 個 FITS 檔案準備批次處理。")

    for file_path in fits_files:
        process_single_fits(file_path, OUTPUT_DIR, my_degrader)

    print("\n🎉 全部 FITS 檔案批次處理完成！帶有絕對物理極值的訓練集準備就緒。")

🔍 總共找到 9 個 FITS 檔案準備批次處理。

🚀 開始處理: M101(1)
✅ 成功儲存 755 張成對切片與極值至 -> processed_dataset\dataset_M101(1).npz

🚀 開始處理: M101(2)
✅ 成功儲存 752 張成對切片與極值至 -> processed_dataset\dataset_M101(2).npz

🚀 開始處理: M101(3)
✅ 成功儲存 903 張成對切片與極值至 -> processed_dataset\dataset_M101(3).npz

🚀 開始處理: M31(1)
✅ 成功儲存 791 張成對切片與極值至 -> processed_dataset\dataset_M31(1).npz

🚀 開始處理: M31(2)
✅ 成功儲存 792 張成對切片與極值至 -> processed_dataset\dataset_M31(2).npz

🚀 開始處理: M31(3)
✅ 成功儲存 871 張成對切片與極值至 -> processed_dataset\dataset_M31(3).npz

🚀 開始處理: M51(1)
✅ 成功儲存 751 張成對切片與極值至 -> processed_dataset\dataset_M51(1).npz

🚀 開始處理: M51(2)
✅ 成功儲存 747 張成對切片與極值至 -> processed_dataset\dataset_M51(2).npz

🚀 開始處理: M51(3)
✅ 成功儲存 771 張成對切片與極值至 -> processed_dataset\dataset_M51(3).npz

🎉 全部 FITS 檔案批次處理完成！帶有絕對物理極值的訓練集準備就緒。
